In [1]:
import nltk
import numpy as np
import pandas as pd
import string as st
import re

from adarank import AdaRank
from metrics import NDCGScorer, APScorer

from sklearn.preprocessing import LabelEncoder
from nltk import PorterStemmer, WordNetLemmatizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
data = pd.read_csv('dataset.csv', encoding='utf-8')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 201 entries, 0 to 200
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   query             201 non-null    object
 1   long_common_name  201 non-null    object
 2   component         201 non-null    object
 3   system            201 non-null    object
 4   property          201 non-null    object
 5   relevance         201 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 9.6+ KB


In [3]:
encoders = {}
for col in ['query', 'system', 'property']:
  le = LabelEncoder()
  data[col+'_cat'] = le.fit_transform(data[col])
  encoders[col] = le
data.head()

,query,long_common_name,component,system,property,relevance,query_cat,system_cat,property_cat
0,glucose in blood,C reactive protein [Mass/volume] in Serum or P...,C reactive protein,Ser/Plas,MCnc,0,1,8,3
1,glucose in blood,Bicarbonate [Moles/volume] in Blood,Bicarbonate,Bld,SCnc,0,1,0,11
2,glucose in blood,Rh [Type] in Blood,Rh,Bld,Type,0,1,0,14
3,glucose in blood,Trimethoprim+Sulfamethoxazole [Susceptibility],Trimethoprim+Sulfamethoxazole,Isolate,Susc,0,1,4,12
4,glucose in blood,Bilirubin.total [Mass/volume] in Serum or Plasma,Bilirubin,Ser/Plas,MCnc,0,1,8,3


In [4]:
def remove_punct(text):
    return ("".join([ch for ch in text if ch not in st.punctuation]))

def tokenize(text):
    text = re.split(r'\s+' ,text)
    return [x.lower() for x in text]

def remove_small_words(text):
    return [x for x in text if len(x) > 3 ]

def remove_stopwords(text):
    return [word for word in text if word not in nltk.corpus.stopwords.words('english')]

def lemmatize(text):
    word_net = WordNetLemmatizer()
    return [word_net.lemmatize(word) for word in text]

def return_sentences(tokens):
    return " ".join([word for word in tokens])

In [5]:
data['text'] = data['long_common_name'] + ' ' + data['component']
data['removed_punc'] = data['text'].apply(lambda x: remove_punct(x))
data['tokens'] = data['removed_punc'].apply(lambda msg : tokenize(msg))
#data['larger_tokens'] = data['tokens'].apply(lambda x : remove_small_words(x))
data['clean_tokens'] = data['tokens'].apply(lambda x : remove_stopwords(x))
data['lemma_words'] = data['clean_tokens'].apply(lambda x : lemmatize(x))
data['clean_text'] = data['lemma_words'].apply(lambda x : return_sentences(x))
data.head().transpose()

,0,1,2,3,4
query,glucose in blood,glucose in blood,glucose in blood,glucose in blood,glucose in blood
long_common_name,C reactive protein [Mass/volume] in Serum or P...,Bicarbonate [Moles/volume] in Blood,Rh [Type] in Blood,Trimethoprim+Sulfamethoxazole [Susceptibility],Bilirubin.total [Mass/volume] in Serum or Plasma
component,C reactive protein,Bicarbonate,Rh,Trimethoprim+Sulfamethoxazole,Bilirubin
system,Ser/Plas,Bld,Bld,Isolate,Ser/Plas
property,MCnc,SCnc,Type,Susc,MCnc
relevance,0,0,0,0,0
query_cat,1,1,1,1,1
system_cat,8,0,0,4,8
property_cat,3,11,14,12,3
text,C reactive protein [Mass/volume] in Serum or P...,Bicarbonate [Moles/volume] in Blood Bicarbonate,Rh [Type] in Blood Rh,Trimethoprim+Sulfamethoxazole [Susceptibility]...,Bilirubin.total [Mass/volume] in Serum or Plas...


In [6]:
tfidf = TfidfVectorizer(max_features=20)
data['FV_document'] = tfidf.fit_transform(data['clean_text']).toarray().tolist()
data['FV_query'] = tfidf.transform(data['query']).toarray().tolist()
data['cosine_similarity'] = data.apply(lambda row: cosine_similarity([row['FV_query']],[row['FV_document']])[0][0], axis=1)
df = data[['query', 'query_cat', 'FV_query', 'FV_document', 'cosine_similarity', 'system_cat', 'property_cat', 'relevance']].copy()
df.head()

,query,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,relevance
0,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,8,3,0
1,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.6802009394700788, 0.0, 0.0, 0.0, ...",0.384027,0,11,0
2,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.318749,0,14,0
3,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,4,12,0
4,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,8,3,0


In [7]:
feature_columns = [
    'cosine_similarity',
    'system_cat',
    'property_cat'
    #'bm25',
    #'jaccard',
    #'term_overlap',
    #'semantic_similarity'
]

df['features'] = df[feature_columns].values.tolist()
df.head()

,query,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,relevance,features
0,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,8,3,0,"[0.0, 8.0, 3.0]"
1,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.6802009394700788, 0.0, 0.0, 0.0, ...",0.384027,0,11,0,"[0.3840266227863685, 0.0, 11.0]"
2,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.318749,0,14,0,"[0.3187485451470555, 0.0, 14.0]"
3,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,4,12,0,"[0.0, 4.0, 12.0]"
4,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,8,3,0,"[0.0, 8.0, 3.0]"


In [8]:
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit

split1 = GroupShuffleSplit(n_splits=1,test_size=0.20,random_state=23)
train_idx, test_idx = next(split1.split(df, groups=df['query_cat']))

df_train = df.iloc[train_idx].copy()
df_test = df.iloc[test_idx].copy()

In [9]:
X_train = csr_matrix(df_train['features'].tolist())
y_train = df_train['relevance'].to_numpy()
qid_train = df_train['query_cat'].to_numpy()

X_test = csr_matrix(df_test['features'].tolist())
y_test = df_test['relevance'].to_numpy()
qid_test = df_test['query_cat'].to_numpy()

X_train.shape, X_test.shape, y_train.shape, y_test.shape, qid_train.shape, qid_test.shape

((134, 3), (67, 3), (134,), (67,), (134,), (67,))

In [10]:
scorer = NDCGScorer(k=10)
#scorer = APScorer()
model = AdaRank(max_iter=100, estop=10, scorer=scorer).fit(X_train, y_train, qid_train)
pred = model.predict(X_test, qid_test)
print(scorer(y_test, pred, qid_test).mean())

0.9328502177967409


In [11]:
#'glucose in blood'
df_query = df_test[df_test['query_cat'] == 1].copy()
X_query = csr_matrix(df_query['features'].tolist())
scores = model.predict(X_query, [1] * len(df_query))
df_query['ranking_score'] = scores
df_query = df_query.sort_values(by='ranking_score', ascending=False)
df_query.head()

,query,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,relevance,features,ranking_score
23,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.290756789709725, 0.0, 0.0, 0.0, 0...",0.865842,9,3,2,"[0.8658415694465108, 9.0, 3.0]",0.476220
28,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.94425642...",0.779370,8,11,1,"[0.7793699722847076, 8.0, 11.0]",0.428660
19,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.93829715...",0.774451,6,11,1,"[0.7744513159008739, 6.0, 11.0]",0.425954
12,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.93829715...",0.774451,11,11,1,"[0.7744513159008739, 11.0, 11.0]",0.425954
18,glucose in blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.88613768...",0.731400,8,11,1,"[0.7313999511518468, 8.0, 11.0]",0.402276


In [12]:
# Lista de documentos ordenados por relevancia dada una consulta
data.loc[df_query.index][:10]

,query,long_common_name,component,system,property,relevance,query_cat,system_cat,property_cat,text,removed_punc,tokens,clean_tokens,lemma_words,clean_text,FV_document,FV_query,cosine_similarity
23,glucose in blood,"Glucose [Mass/volume] in Serum, Plasma or Blood",Glucose,Ser/Plas/Bld,MCnc,2,1,9,3,"Glucose [Mass/volume] in Serum, Plasma or Bloo...",Glucose Massvolume in Serum Plasma or Blood Gl...,"[glucose, massvolume, in, serum, plasma, or, b...","[glucose, massvolume, serum, plasma, blood, gl...","[glucose, massvolume, serum, plasma, blood, gl...",glucose massvolume serum plasma blood glucose,"[0.0, 0.0, 0.290756789709725, 0.0, 0.0, 0.0, 0...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.865842
28,glucose in blood,Glucose [Moles/volume] in Serum or Plasma --3 ...,Glucose^3H post 100 g glucose PO,Ser/Plas,SCnc,1,1,8,11,Glucose [Moles/volume] in Serum or Plasma --3 ...,Glucose Molesvolume in Serum or Plasma 3 hours...,"[glucose, molesvolume, in, serum, or, plasma, ...","[glucose, molesvolume, serum, plasma, 3, hours...","[glucose, molesvolume, serum, plasma, 3, hour,...",glucose molesvolume serum plasma 3 hour post 1...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.94425642...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.779370
19,glucose in blood,Glucose [Moles/volume] in Pleural fluid,Glucose,Plr fld,SCnc,1,1,6,11,Glucose [Moles/volume] in Pleural fluid Glucose,Glucose Molesvolume in Pleural fluid Glucose,"[glucose, molesvolume, in, pleural, fluid, glu...","[glucose, molesvolume, pleural, fluid, glucose]","[glucose, molesvolume, pleural, fluid, glucose]",glucose molesvolume pleural fluid glucose,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.93829715...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.774451
12,glucose in blood,Glucose [Moles/volume] in Urine,Glucose,Urine,SCnc,1,1,11,11,Glucose [Moles/volume] in Urine Glucose,Glucose Molesvolume in Urine Glucose,"[glucose, molesvolume, in, urine, glucose]","[glucose, molesvolume, urine, glucose]","[glucose, molesvolume, urine, glucose]",glucose molesvolume urine glucose,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.93829715...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.774451
18,glucose in blood,Glucose [Moles/volume] in Serum or Plasma,Glucose,Ser/Plas,SCnc,1,1,8,11,Glucose [Moles/volume] in Serum or Plasma Glucose,Glucose Molesvolume in Serum or Plasma Glucose,"[glucose, molesvolume, in, serum, or, plasma, ...","[glucose, molesvolume, serum, plasma, glucose]","[glucose, molesvolume, serum, plasma, glucose]",glucose molesvolume serum plasma glucose,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.88613768...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.731400
55,glucose in blood,Fasting glucose [Mass or Moles/volume] in Seru...,Glucose^post CFst,Ser/Plas,MSCnc,0,1,8,6,Fasting glucose [Mass or Moles/volume] in Seru...,Fasting glucose Mass or Molesvolume in Serum o...,"[fasting, glucose, mass, or, molesvolume, in, ...","[fasting, glucose, mass, molesvolume, serum, p...","[fasting, glucose, mass, molesvolume, serum, p...",fasting glucose mass molesvolume serum plasma ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.69105611...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.570384
52,glucose in blood,Lymphocytes/100 leukocytes in Blood,Lymphocytes/100 leukocytes,Bld,NFr,0,1,0,8,Lymphocytes/100 leukocytes in Blood Lymphocyte...,Lymphocytes100 leukocytes in Blood Lymphocytes...,"[lymphocytes100, leukocytes, in, blood, lympho...","[lymphocytes100, leukocytes, blood, lymphocyte...","[lymphocytes100, leukocyte, blood, lymphocytes...",lymphocytes100 leukocyte blood lymphocytes100 ...,"[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.564578
41,glucose in blood,Chloride [Moles/volume] in Blood,Chloride,Bld,SCnc,0,1,0,11,Chloride [Moles/volume] in Blood Chloride,Chloride Molesvolume in Blood Chloride,"[chloride, molesvolume, in, blood, chloride]","[chloride, molesvolume, blood, chloride]","[chloride, molesvolume, 

### Nueva consulta con los documentos antiguos

In [13]:
new_query = ['sodium in serum']
df_new = pd.DataFrame([new_query], columns=['query'])
df_new['query_cat'] = encoders['query'].classes_.size
df_new['FV_query'] = tfidf.transform(df_new['query']).toarray().tolist()
df_new.head()

,query,query_cat,FV_query
0,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [14]:
df_new = pd.concat([df_new] * len(data['FV_document']), ignore_index=True)
df_new = pd.concat([df_new, data[['system_cat', 'property_cat', 'FV_document']]], axis=1)
df_new.head()

,query,query_cat,FV_query,system_cat,property_cat,FV_document
0,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",8,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,11,"[0.0, 0.0, 0.6802009394700788, 0.0, 0.0, 0.0, ..."
2,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,14,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ..."
3,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",4,12,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",8,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [15]:
df_new['cosine_similarity'] = df_new.apply(lambda row: cosine_similarity([row['FV_query']],[row['FV_document']])[0][0], axis=1)
df_new = df_new[['query', 'query_cat', 'FV_query', 'FV_document', 'cosine_similarity', 'system_cat', 'property_cat']].copy()
df_new['features'] = df_new[feature_columns].values.tolist()
df_new.head()

,query,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,features
0,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.142983,8,3,"[0.14298329947782673, 8.0, 3.0]"
1,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.6802009394700788, 0.0, 0.0, 0.0, ...",0.000000,0,11,"[0.0, 0.0, 11.0]"
2,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.000000,0,14,"[0.0, 0.0, 14.0]"
3,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000,4,12,"[0.0, 4.0, 12.0]"
4,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.508035,8,3,"[0.5080352218599479, 8.0, 3.0]"


In [16]:
X_new = csr_matrix(df_new['features'].tolist())
scores = model.predict(X_new, [3] * len(df_new))
df_new['ranking_score'] = scores
df_new = df_new.sort_values(by='ranking_score', ascending=False)
df_new.head()

,query,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,features,ranking_score
115,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107,8,0,"[0.7071067811865476, 8.0, 0.0]",0.388914
182,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107,8,4,"[0.7071067811865476, 8.0, 4.0]",0.388914
53,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107,8,0,"[0.7071067811865476, 8.0, 0.0]",0.388914
124,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107,8,1,"[0.7071067811865476, 8.0, 1.0]",0.388914
122,sodium in serum,3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107,8,1,"[0.7071067811865476, 8.0, 1.0]",0.388914


In [17]:
# Lista de documentos ordenados por relevancia dada una consulta que el modelo no vio en el entrenamiento
data.loc[df_new.index][:10]

,query,long_common_name,component,system,property,relevance,query_cat,system_cat,property_cat,text,removed_punc,tokens,clean_tokens,lemma_words,clean_text,FV_document,FV_query,cosine_similarity
115,bilirubin in plasma,Choriogonadotropin [Units/volume] in Serum or ...,Choriogonadotropin,Ser/Plas,ACnc,0,0,8,0,Choriogonadotropin [Units/volume] in Serum or ...,Choriogonadotropin Unitsvolume in Serum or Pla...,"[choriogonadotropin, unitsvolume, in, serum, o...","[choriogonadotropin, unitsvolume, serum, plasm...","[choriogonadotropin, unitsvolume, serum, plasm...",choriogonadotropin unitsvolume serum plasma ch...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107
182,white blood cells count,Albumin/Globulin [Mass ratio] in Serum or Plasma,Albumin/Globulin,Ser/Plas,MCrto,0,2,8,4,Albumin/Globulin [Mass ratio] in Serum or Plas...,AlbuminGlobulin Mass ratio in Serum or Plasma ...,"[albuminglobulin, mass, ratio, in, serum, or, ...","[albuminglobulin, mass, ratio, serum, plasma, ...","[albuminglobulin, mass, ratio, serum, plasma, ...",albuminglobulin mass ratio serum plasma albumi...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000
53,glucose in blood,Choriogonadotropin [Units/volume] in Serum or ...,Choriogonadotropin,Ser/Plas,ACnc,0,1,8,0,Choriogonadotropin [Units/volume] in Serum or ...,Choriogonadotropin Unitsvolume in Serum or Pla...,"[choriogonadotropin, unitsvolume, in, serum, o...","[choriogonadotropin, unitsvolume, serum, plasm...","[choriogonadotropin, unitsvolume, serum, plasm...",choriogonadotropin unitsvolume serum plasma ch...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.000000
124,bilirubin in plasma,Amylase [Enzymatic activity/volume] in Serum o...,Amylase,Ser/Plas,CCnc,0,0,8,1,Amylase [Enzymatic activity/volume] in Serum o...,Amylase Enzymatic activityvolume in Serum or P...,"[amylase, enzymatic, activityvolume, in, serum...","[amylase, enzymatic, activityvolume, serum, pl...","[amylase, enzymatic, activityvolume, serum, pl...",amylase enzymatic activityvolume serum plasma ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107
122,bilirubin in plasma,Alkaline phosphatase [Enzymatic activity/volum...,Alkaline phosphatase,Ser/Plas,CCnc,0,0,8,1,Alkaline phosphatase [Enzymatic activity/volum...,Alkaline phosphatase Enzymatic activityvolume ...,"[alkaline, phosphatase, enzymatic, activityvol...","[alkaline, phosphatase, enzymatic, activityvol...","[alkaline, phosphatase, enzymatic, activityvol...",alkaline phosphatase enzymatic activityvolume ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.707107
151,white blood cells count,Amylase [Enzymatic activity/volume] in Serum o...,Amylase,Ser/Plas,CCnc,0,2,8,1,Amylase [Enzymatic activity/volume] in Serum o...,Amylase Enzymatic activityvolume in Serum or P...,"[amylase, enzymatic, activityvolume, in, serum...","[amylase, enzymatic, activityvolume, serum, pl...","[amylase, enzymatic, activityvolume, serum, pl...",amylase enzymatic activityvolume serum plasma ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.000000
40,glucose in blood,Alkaline phosphatase [Enzymatic activity/volum...,Alkaline phosphatase,Ser/Plas,CCnc,0,1,8,1,Alkaline phosphatase [Enzymatic activity/volum...,Alkaline phosphatase Enzymatic activityvolume ...,"[alkaline, phosphatase, enzymatic, activityvol...","[alkaline, phosphatase, enzymatic, activityvol...","[alkaline, phosphatase, enzymatic, activityvol...",alkaline phosphatase enzymatic activityvolume ...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",0.000000
164,white blood cells count,Choriogonadotropin [Units/volume] in Serum or ...,Choriogona

### Anterior consulta con nuevos términos

In [18]:
df_extended = pd.read_csv('dataset_extended.csv', encoding='utf-8')
for col in ['query', 'system', 'property']:
    mapping = {cls: i for i, cls in enumerate(encoders[col].classes_)}
    next_label = len(le.classes_)
    encoded = []

    for value in df_extended[col]:
        if value not in mapping:
            mapping[value] = next_label
            next_label += 1
        encoded.append(mapping[value])

    df_extended[col + '_cat'] = encoded
df_extended.head()

,query,long_common_name,component,system,property,query_cat,system_cat,property_cat
0,glucose in blood,Glucose [Moles/volume] in Blood,Glucose,Bld,SCnc,1,0,11
1,glucose in blood,Glucose [Mass/volume] in Blood,Glucose,Bld,MCnc,1,0,3
2,glucose in blood,Glucose [Mass/volume] in Capillary blood --bas...,Glucose^baseline,BldC,MCnc,1,16,3
3,glucose in blood,Glucose [Mass/volume] in Venous blood,Glucose,BldV,MCnc,1,17,3
4,glucose in blood,Glucose [Moles/volume] in Arterial blood,Glucose,BldA,SCnc,1,18,11


In [19]:
df_extended['text'] = df_extended['long_common_name'] + ' ' + df_extended['component']
df_extended['removed_punc'] = df_extended['text'].apply(lambda x: remove_punct(x))
df_extended['tokens'] = df_extended['removed_punc'].apply(lambda msg : tokenize(msg))
#df_extended['larger_tokens'] = df_extended['tokens'].apply(lambda x : remove_small_words(x))
df_extended['clean_tokens'] = df_extended['tokens'].apply(lambda x : remove_stopwords(x))
df_extended['lemma_words'] = df_extended['clean_tokens'].apply(lambda x : lemmatize(x))
df_extended['clean_text'] = df_extended['lemma_words'].apply(lambda x : return_sentences(x))

In [20]:
df_extended['FV_document'] = tfidf.transform(df_extended['clean_text']).toarray().tolist()
df_extended['FV_query'] = tfidf.transform(df_extended['query']).toarray().tolist()
df_extended['cosine_similarity'] = df_extended.apply(lambda row: cosine_similarity([row['FV_query']],[row['FV_document']])[0][0], axis=1)
df_extended = df_extended[['query','long_common_name','query_cat', 'FV_query', 'FV_document', 'cosine_similarity', 'system_cat', 'property_cat']].copy()
df_extended['features'] = df_extended[feature_columns].values.tolist()
df_extended.head()

,query,long_common_name,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,features
0,glucose in blood,Glucose [Moles/volume] in Blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30556004457702945, 0.0, 0.0, 0.0,...",0.909924,0,11,"[0.9099240255776974, 0.0, 11.0]"
1,glucose in blood,Glucose [Mass/volume] in Blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,0,3,"[0.912416100376007, 0.0, 3.0]"
2,glucose in blood,Glucose [Mass/volume] in Capillary blood --bas...,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.4856317803800905, 0.0, 0.0, 0.0, ...",0.860167,16,3,"[0.8601674302013781, 16.0, 3.0]"
3,glucose in blood,Glucose [Mass/volume] in Venous blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,17,3,"[0.912416100376007, 17.0, 3.0]"
4,glucose in blood,Glucose [Moles/volume] in Arterial blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30556004457702945, 0.0, 0.0, 0.0,...",0.909924,18,11,"[0.9099240255776974, 18.0, 11.0]"


In [21]:
# glucose in blood
df_new = df_extended.iloc[:33].copy()
X_new = csr_matrix(df_new['features'].tolist())
scores = model.predict(X_new, [1] * len(df_new))
df_new['ranking_score'] = scores
df_new = df_new.sort_values(by='ranking_score', ascending=False)
df_new.head()

,query,long_common_name,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,features,ranking_score
13,glucose in blood,Blood glucose monitor attachment,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...",1.000000,14,16,"[1.0000000000000002, 14.0, 16.0]",0.550008
1,glucose in blood,Glucose [Mass/volume] in Blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,0,3,"[0.912416100376007, 0.0, 3.0]",0.501836
26,glucose in blood,Glucose [Mass/volume] in Mixed venous blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,21,3,"[0.912416100376007, 21.0, 3.0]",0.501836
3,glucose in blood,Glucose [Mass/volume] in Venous blood,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,17,3,"[0.912416100376007, 17.0, 3.0]",0.501836
19,glucose in blood,Glucose mean value [Mass/volume] in Blood Esti...,1,"[0.0, 0.0, 0.5645782010909166, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.30639690399062425, 0.0, 0.0, 0.0,...",0.912416,0,3,"[0.912416100376007, 0.0, 3.0]",0.501836


### Nueva consulta con nuevos términos

In [22]:
# creatinine in serum
df_new = df_extended.iloc[-100:].copy()
X_new = csr_matrix(df_new['features'].tolist())
scores = model.predict(X_new, [5] * len(df_new))
df_new['ranking_score'] = scores
df_new = df_new.sort_values(by='ranking_score', ascending=False)
df_new.head()

,query,long_common_name,query_cat,FV_query,FV_document,cosine_similarity,system_cat,property_cat,features,ranking_score
192,creatinine in serum,PhenX - serum creatinine assay for kidney func...,17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,14,22,"[1.0, 14.0, 22.0]",0.550008
193,creatinine in serum,Creatinine in peritoneal fluid/Creatinine in s...,17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,35,35,"[1.0, 35.0, 35.0]",0.550008
194,creatinine in serum,Creatinine in pleural fluid/Creatinine in serum,17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,36,35,"[1.0, 36.0, 35.0]",0.550008
283,creatinine in serum,"HEDIS 2010,2011 Codes to identify physiologic ...",17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,14,22,"[1.0, 14.0, 22.0]",0.550008
284,creatinine in serum,Deprecated PhenX measure - serum creatinine as...,17,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,14,22,"[1.0, 14.0, 22.0]",0.550008
